# الدرس 11 - بروتوكول سياق النموذج (MCP)

يعد **بروتوكول سياق النموذج (MCP)** معيارًا مفتوحًا يمكّن الوكلاء من اكتشاف الأدوات والموارد ومصادر البيانات واستخدامها ديناميكيًا في وقت التشغيل. بدلاً من ترميز الأدوات داخل الوكيل بشكل ثابت، يتيح MCP للوكلاء الاتصال بخوادم خارجية تعرض إمكانيات عند الطلب.

في هذا الدرس، ستتعلم:
- ما هو MCP ولماذا يهم لأنظمة الوكلاء
- كيف تعمل بنية عميل-خادم MCP
- كيفية بناء وكلاء يستخدمون اكتشاف الأدوات بنمط MCP


## الإعداد

**المتطلبات المسبقة:**
- مشروع Azure AI Foundry مع نموذج منشور
- قم بتشغيل `az login` للمصادقة


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## ما هو بروتوكول سياق النموذج (MCP)؟

MCP يعرف طريقة قياسية تتيح لوكلاء الذكاء الاصطناعي الاكتشاف والتفاعل مع الأدوات ومصادر البيانات الخارجية:

- **MCP Server**: يعرض الأدوات والموارد والمطالبات عبر بروتوكول قياسي
- **MCP Client**: بيئة تشغيل الوكيل التي تتصل بالخوادم وتكتشف القدرات المتاحة
- **Dynamic Discovery**: لا يحتاج الوكلاء إلى أدوات مضمنة بشكل ثابت — فهم يكتشفون ما هو متاح أثناء وقت التشغيل

هذا مفيد لبناء أنظمة وكلاء قابلة للتوسعة حيث يمكن إضافة قدرات جديدة دون تعديل كود الوكيل.


## كيف يعمل MCP

```
┌─────────────┐     discover      ┌─────────────────┐
│  MCP Client  │ ──────────────► │   MCP Server     │
│  (Agent)     │                  │  (Tool Provider) │
│              │ ◄────────────── │                   │
│              │   tool results   │  • list_tools()  │
│              │                  │  • call_tool()   │
└─────────────┘                  │  • resources     │
                                  └─────────────────┘
```

1. يتصل الوكيل (MCP client) بـMCP server
2. يرد الخادم بقائمة من الأدوات المتاحة ومخططاتها
3. ثم يمكن للوكيل استدعاء أي أداة تم اكتشافها أثناء استدلاله
4. تعود النتائج عبر نفس البروتوكول


## محاكاة اكتشاف أدوات MCP

نظرًا لأن خادم MCP الحقيقي يتطلب عملية خادم قيد التشغيل، سنعرض النمط باستخدام دوال `@tool` التي تحاكي ما ستوفره خدمة الإقامة المتصلة بـ MCP.

في بيئة الإنتاج، ستُكتشف هذه الأدوات ديناميكيًا من خادم MCP بدلاً من تعريفها محليًا.


In [ ]:
@tool(approval_mode="never_require")
def search_accommodations(
    location: Annotated[str, "The city to search for accommodations"],
    check_in: Annotated[str, "Check-in date (YYYY-MM-DD)"],
    check_out: Annotated[str, "Check-out date (YYYY-MM-DD)"],
    guests: Annotated[int, "Number of guests"] = 2
) -> str:
    """Search for accommodations (simulating an MCP-connected Airbnb tool).
    In production, this would be discovered via MCP from an accommodation service."""
    listings = {
        "Tokyo": [
            {"name": "Shinjuku Modern Apartment", "price": 120, "rating": 4.8},
            {"name": "Traditional Ryokan in Asakusa", "price": 200, "rating": 4.9},
            {"name": "Shibuya Studio", "price": 85, "rating": 4.5},
        ],
        "Paris": [
            {"name": "Le Marais Charming Flat", "price": 150, "rating": 4.7},
            {"name": "Montmartre Artist Loft", "price": 110, "rating": 4.6},
        ],
        "Barcelona": [
            {"name": "Gothic Quarter Penthouse", "price": 130, "rating": 4.8},
            {"name": "Barceloneta Beach Flat", "price": 95, "rating": 4.4},
        ],
    }
    results = listings.get(location, [])
    if not results:
        return f"No accommodations found in {location}"
    output = f"Accommodations in {location} ({check_in} to {check_out}, {guests} guests):\n"
    for listing in results:
        output += f"  - {listing['name']}: ${listing['price']}/night (★{listing['rating']})\n"
    return output


@tool(approval_mode="never_require")
def get_local_experiences(
    location: Annotated[str, "The city to find experiences in"],
    interest: Annotated[str, "Type of experience (food, culture, adventure, etc.)"] = "all"
) -> str:
    """Get local experiences and activities (simulating an MCP-connected tourism tool)."""
    experiences = {
        "Tokyo": {
            "food": ["Tsukiji Market Tour ($45)", "Ramen Making Class ($60)", "Sake Tasting ($35)"],
            "culture": ["Tea Ceremony ($50)", "Samurai Museum ($15)", "Sumo Tournament ($80)"],
            "adventure": ["Mt. Fuji Day Trip ($120)", "Go-kart City Tour ($80)"],
        },
        "Paris": {
            "food": ["Wine & Cheese Tasting ($55)", "Cooking Class ($90)", "Market Tour ($40)"],
            "culture": ["Louvre Guided Tour ($35)", "Montmartre Art Walk ($25)"],
        },
    }
    city_exp = experiences.get(location, {})
    if not city_exp:
        return f"No experiences found in {location}"
    if interest != "all" and interest in city_exp:
        items = city_exp[interest]
        return f"{interest.title()} experiences in {location}:\n" + "\n".join(f"  - {e}" for e in items)
    output = f"All experiences in {location}:\n"
    for cat, items in city_exp.items():
        output += f"\n  {cat.title()}:\n"
        for item in items:
            output += f"    - {item}\n"
    return output

## بناء وكيل باستخدام أدوات على نمط MCP


In [ ]:
agent = await provider.create_agent(
    tools=[search_accommodations, get_local_experiences],
    name="AccommodationAgent",
    instructions="""You are an accommodation and travel experiences specialist powered by MCP-connected services.

Help travelers find the perfect place to stay and things to do. When searching:
1. Use the search_accommodations tool to find listings
2. Use the get_local_experiences tool to suggest activities
3. Compare options and make personalized recommendations
4. Consider the traveler's budget, interests, and travel style""",
)

response = await agent.run(
    "I'm visiting Tokyo for 5 nights in April with my partner. We love traditional Japanese culture and food. "
    "Find us a place to stay and suggest some experiences.",
    )
print(response)

## MCP في الإنتاج

في بيئة الإنتاج، يتيح MCP أنماطًا قوية:

- **اكتشاف الأدوات الديناميكي**: يتصل الوكلاء بخوادم MCP ويكتشفون الأدوات أثناء وقت التشغيل
- **معمارية منفصلة**: يمكن لمزودي الأدوات التحديث بشكل مستقل عن الوكيل
- **المشاركة عبر المؤسسات**: يمكن للفرق إتاحة الإمكانيات عبر خوادم MCP بحيث يمكن لأي وكيل استخدامها
- **دعم Microsoft Agent Framework**: يحتوي MAF على دعم مدمج لعميل MCP عبر تكامل `mcp`

لاستخدام خادم MCP حقيقي مع MAF، يمكنك الاتصال عبر `hosted_mcp_tool()` أو عبر تكامل عميل MCP.

**تعرف على المزيد:**
- [MCP Specification](https://modelcontextprotocol.io/)
- [Microsoft Agent Framework MCP Support](https://github.com/microsoft/agent-framework/tree/main/python/samples/02-agents/mcp)


## الملخص

في هذا الدرس، تعلّمت:
- **MCP** هو معيار مفتوح لاكتشاف الأدوات بشكل ديناميكي بين الوكلاء ومزودي الأدوات
- تتيح **معمارية العميل-الخادم** للوكلاء اكتشاف الإمكانيات أثناء وقت التشغيل
- MCP يمكّن **أنظمة وكلاء قابلة للتمديد ومنفصلة** حيث يمكن إضافة الأدوات دون تغييرات في الشيفرة
- يوفر Microsoft Agent Framework **دعم مدمج لـMCP** للاستخدام في الإنتاج


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
إخلاء المسؤولية:
تمت ترجمة هذا المستند باستخدام خدمة الترجمة الآلية Co-op Translator (https://github.com/Azure/co-op-translator). بينما نسعى لتحقيق الدقة، يُرجى العلم أن الترجمات الآلية قد تحتوي على أخطاء أو عدم دقة. يجب اعتبار المستند الأصلي بلغته الأصلية المصدر المعتمد. بالنسبة للمعلومات الحرجة، يُنصَح بالاستعانة بترجمة بشرية محترفة. لا نتحمّل أي مسؤولية عن أي سوء فهم أو تفسير ناتج عن استخدام هذه الترجمة.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
